In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
df = pd.read_csv('cleaned_cars.csv')
print("Data loaded successfully!")
print(f"Original shape: {df.shape}")
print(f"Original columns: {df.columns.tolist()}")

In [ ]:
# Price per KM = AskPrice / kmDriven

# 1. Handle zero kmDriven values
df['kmDriven'] = df['kmDriven'].replace(0, np.nan)

# 2. Replace missing kmDriven values with median
median_km = df['kmDriven'].median()
df['kmDriven'] = df['kmDriven'].fillna(median_km)

# 3. Create PricePerKM feature
df['PricePerKM'] = df['AskPrice'] / df['kmDriven']

# 4. Basic statistics
print("✅ Created PricePerKM feature")
print(f"Mean Price per KM   : ₹{df['PricePerKM'].mean():.2f}")
print(f"Median Price per KM : ₹{df['PricePerKM'].median():.2f}")
print(f"Minimum             : ₹{df['PricePerKM'].min():.2f}")
print(f"Maximum             : ₹{df['PricePerKM'].max():.2f}")

# 5. Check extreme values
print("\n🔍 Top 10 Price per KM values:")
print(df.nlargest(10, 'PricePerKM')[['AskPrice', 'kmDriven', 'PricePerKM']])

# 6. Use 99th percentile only for visualization
upper_limit = df['PricePerKM'].quantile(0.99)

plot_data = df[df['PricePerKM'] <= upper_limit]['PricePerKM']

# 7. Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(
    plot_data,
    bins=50,
    edgecolor='black'
)

axes[0].set_title('Price per KM Distribution')
axes[0].set_xlabel('₹ per KM')
axes[0].set_ylabel('Count')

# Median line
median_price_km = df['PricePerKM'].median()

axes[0].axvline(median_price_km,linestyle='--',label=f'Median: ₹{median_price_km:.2f}')

axes[0].legend()


# Box plot
axes[1].boxplot(plot_data)

axes[1].set_title('Price per KM Box Plot (99th Percentile)')
axes[1].set_ylabel('₹ per KM')

plt.tight_layout()
plt.show()

In [ ]:
# Create price categories
bins = [0, 300000, 600000, 1000000, 2000000, float('inf')]
labels = ['Budget', 'Economy', 'Mid-Range', 'Premium', 'Luxury']
df['PriceCategory'] = pd.cut(df['AskPrice'], bins=bins, labels=labels)

print("✅ Created PriceCategory feature")
print("\nPrice Category Distribution:")
category_counts = df['PriceCategory'].value_counts()
for cat, count in category_counts.items():
    print(f"   {cat}: {count:,} cars ({count/len(df)*100:.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
bars = axes[0].bar(category_counts.index, category_counts.values, 
                   color=['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c'])
axes[0].set_title('Price Categories')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                 f'{int(height)}', ha='center', va='bottom')

# Pie chart
axes[1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c'])
axes[1].set_title('Price Category Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# KM per Year = kmDriven / Age
# Handle zero age to avoid division by zero
df['Age'] = df['Age'].replace(0, 1)
df['KMPerYear'] = df['kmDriven'] / df['Age']

print("✅ Created KMPerYear feature")
print(f"Average KM per Year: {df['KMPerYear'].mean():,.0f} km")
print(f"Range: {df['KMPerYear'].min():,.0f} - {df['KMPerYear'].max():,.0f} km")

# Create usage categories
bins = [0, 5000, 10000, 15000, 20000, float('inf')]
labels = ['Very Low', 'Low', 'Normal', 'High', 'Very High']
df['UsageCategory'] = pd.cut(df['KMPerYear'], bins=bins, labels=labels)

print("\n✅ Created UsageCategory feature")
print("\nUsage Category Distribution:")
usage_counts = df['UsageCategory'].value_counts()
for cat, count in usage_counts.items():
    print(f"   {cat}: {count:,} cars ({count/len(df)*100:.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df['KMPerYear'], bins=50, color='#2ecc71', edgecolor='black')
axes[0].set_title('KM per Year Distribution')
axes[0].set_xlabel('KM per Year')
axes[0].axvline(df['KMPerYear'].median(), color='red', linestyle='--', 
                label=f'Median: {df["KMPerYear"].median():,.0f}')
axes[0].legend()

bars = axes[1].bar(usage_counts.index, usage_counts.values, color='#f39c12')
axes[1].set_title('Usage Categories')
axes[1].set_xlabel('Usage')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{int(height)}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Value Score (0-100)
# Lower age, lower km, lower price = better value

# Normalize function
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

# Create individual scores (1 - normalized because lower is better)
age_score = 1 - normalize(df['Age'])
km_score = 1 - normalize(df['kmDriven'])
price_score = 1 - normalize(df['AskPrice'])

# Composite value score
df['ValueScore'] = (age_score + km_score + price_score) / 3 * 100

print("✅ Created ValueScore feature")
print(f"Average Value Score: {df['ValueScore'].mean():.1f}")
print(f"Range: {df['ValueScore'].min():.1f} - {df['ValueScore'].max():.1f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Value Score Distribution
axes[0].hist(df['ValueScore'], bins=50, color='#9b59b6', edgecolor='black')
axes[0].set_title('Value Score Distribution')
axes[0].set_xlabel('Score (0-100)')
axes[0].axvline(df['ValueScore'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["ValueScore"].mean():.1f}')
axes[0].legend()

# Value Score by Category
cat_score = df.groupby('PriceCategory')['ValueScore'].mean()
bars = axes[1].bar(cat_score.index, cat_score.values, color='#3498db')
axes[1].set_title('Average Value Score by Price Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Value Score')
axes[1].tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{height:.1f}', ha='center', va='bottom')

# Value Score vs Price scatter
axes[2].scatter(df['AskPrice'], df['ValueScore'], alpha=0.4, s=10, color='#e74c3c')
axes[2].set_title('Value Score vs Price')
axes[2].set_xlabel('Price (₹)')
axes[2].set_ylabel('Value Score')

plt.tight_layout()
# plt.savefig('value_score_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 value for money cars
top_value = df.nlargest(10, 'ValueScore')[
    ['Brand', 'model', 'Year', 'Age', 'kmDriven', 'AskPrice', 'ValueScore']
].round(2)

print("=" * 50)
print("TOP 10 VALUE FOR MONEY CARS")
print("=" * 50)
print(top_value.to_string(index=False))

# %%
# Best value cars by category
print("\n" + "=" * 50)
print("BEST VALUE CARS BY PRICE CATEGORY")
print("=" * 50)

for category in df['PriceCategory'].dropna().unique():
    cat_cars = df[df['PriceCategory'] == category]
    best_car = cat_cars.nlargest(1, 'ValueScore').iloc[0]
    print(f"\n{category}:")
    print(f"   {best_car['Brand']} {best_car['model']} ({best_car['Year']})")
    print(f"   Price: ₹{best_car['AskPrice']:,.0f}")
    print(f"   KM: {best_car['kmDriven']:,.0f}")
    print(f"   Value Score: {best_car['ValueScore']:.1f}")

# %%
# Save enhanced dataset
# df.to_csv('enhanced_cars.csv', index=False)
print("\n✅ Enhanced data saved as 'enhanced_cars.csv'")

# Show all new features
print("\n" + "=" * 50)
print("ALL FEATURES IN DATASET")
print("=" * 50)
print(f"Original features: {df.shape[1] - 4} columns")
print(f"New features: 4 columns")
print(f"Total: {df.shape[1]} columns")
print("\nNew Features Created:")
print("  1. PricePerKM - Price per kilometer driven")
print("  2. PriceCategory - Budget to Luxury")
print("  3. KMPerYear - Usage intensity per year")
print("  4. UsageCategory - Very Low to Very High usage")
print("  5. ValueScore - Overall value rating (0-100)")

Day 5 Summary

What I Did Today:
 - ✅ Created PricePerKM feature
 - ✅ Created PriceCategory (Budget to Luxury)
 - ✅ Created KMPerYear and UsageCategory
 - ✅ Built composite ValueScore
 - ✅ Identified top 10 value cars
 - ✅ Found best value cars by category
 - ✅ Saved enhanced dataset
 
Key Features Created:
 1. **PricePerKM**: How much car costs per kilometer driven
 2. **PriceCategory**: Budget, Economy, Mid-Range, Premium, Luxury
 3. **KMPerYear**: Average usage intensity
 4. **UsageCategory**: Very Low to Very High usage
 5. **ValueScore**: Composite score (0-100) combining age, km, and price
 
Key Findings:
 - Average price per KM: ₹XX
 - Most cars are in Economy/Mid-Range category
 - Average usage: XX,XXX km per year
 - Best value cars are 3-5 years old with moderate KM